# T-tests: creative specialisation and local context

## Purpose
Test whether TTWAs with above-average creative employment differ from other
TTWAs on socio-economic and urban-rural characteristics. The results form
the t-test table in the ERP appendix.

## Input
- `data/processed/nesta_ball.csv`
- `data/processed/census_filtered.csv`

## Main steps
- Calculate the overall creative employment LQ for each TTWA
- Merge with the census variables by TTWA name (173 TTWAs)
- Split TTWAs into LQ > 1 (17) and LQ ≤ 1 (156)
- For each of 12 contextual variables (qualifications, occupation,
  self-employment, economic inactivity, tenure, population density,
  urbanity and median house price):
  - Descriptive statistics (mean, SD, min, max)
  - Group means and difference (LQ > 1 minus LQ ≤ 1)
  - Welch two-sample t-test (unequal variances)
  - Pearson correlation with the continuous creative LQ

## Output
Appendix table of descriptive statistics, group differences (with
significance stars) and correlations. 

In [1]:
import pandas as pd

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DATA = ROOT / "data" / "raw"
PROCESSED_DATA = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"

In [3]:
nesta = pd.read_csv(PROCESSED_DATA / "nesta_ball.csv")
census = pd.read_csv(PROCESSED_DATA / "census_filtered.csv")

In [4]:
[col for col in nesta.columns if "LQ" in col]

['Advertising and marketing: Employment 2011-2014_LQ',
 'Architecture: Employment 2011-2014_LQ',
 'Design: product, graphic and fashion design: Employment 2011-2014_LQ',
 'Film, TV, video, radio and photography: Employment 2011-2014_LQ',
 'IT, software and computer services: Employment 2011-2014_LQ',
 'Music, performing and visual arts: Employment 2011-2014_LQ',
 'Publishing: Employment 2011-2014_LQ',
 'creative_LQ']

In [5]:
[col for col in nesta.columns if "Employment" in col or "creative" in col.lower()]

['All creative industries: Employment 2011-2014',
 'not_creative: Employment 2011-2014',
 'Advertising and marketing: Employment 2011-2014',
 'Architecture: Employment 2011-2014',
 'Design: product, graphic and fashion design: Employment 2011-2014',
 'Film, TV, video, radio and photography: Employment 2011-2014',
 'IT, software and computer services: Employment 2011-2014',
 'Music, performing and visual arts: Employment 2011-2014',
 'Publishing: Employment 2011-2014',
 'Advertising and marketing: Employment 2011-2014_share',
 'Architecture: Employment 2011-2014_share',
 'Design: product, graphic and fashion design: Employment 2011-2014_share',
 'Film, TV, video, radio and photography: Employment 2011-2014_share',
 'IT, software and computer services: Employment 2011-2014_share',
 'Music, performing and visual arts: Employment 2011-2014_share',
 'Publishing: Employment 2011-2014_share',
 'Advertising and marketing: Employment 2011-2014_LQ',
 'Architecture: Employment 2011-2014_LQ',
 'De

In [6]:
# Make TTWA a normal column rather than the index
nesta = nesta.reset_index()

nesta.head()

,index,ttwa,Region / Nation,All creative industries: Employment 2011-2014,not_creative: Employment 2011-2014,Advertising and marketing: Employment 2011-2014,Architecture: Employment 2011-2014,"Design: product, graphic and fashion design: Employment 2011-2014","Film, TV, video, radio and photography: Employment 2011-2014","IT, software and computer services: Employment 2011-2014",...,Advertising and marketing: Employment 2011-2014_LQ,Architecture: Employment 2011-2014_LQ,"Design: product, graphic and fashion design: Employment 2011-2014_LQ","Film, TV, video, radio and photography: Employment 2011-2014_LQ","IT, software and computer services: Employment 2011-2014_LQ","Music, performing and visual arts: Employment 2011-2014_LQ",Publishing: Employment 2011-2014_LQ,creative_share,creative_LQ,creative_group
0,0,Aberystwyth,Wales,376.00,19789.47368,30.50,29.50,8.25,21.50,96.25,...,0.326870,0.728732,0.238555,0.191961,0.324471,0.878220,1.297171,0.018646,0.503495,False
1,1,Andover,South East,975.25,27864.28571,76.25,31.75,78.00,92.25,352.00,...,0.571394,0.548416,1.577064,0.575920,0.829732,0.599515,2.026933,0.033816,0.913152,False
2,2,Ashford,South East,1467.75,41935.71429,103.75,67.75,84.00,74.50,901.75,...,0.516592,0.777570,1.128490,0.309041,1.412359,0.914428,0.448141,0.033816,0.913152,False
3,3,Banbury,South East,1745.50,37945.65217,199.25,137.75,134.50,198.50,885.25,...,1.084898,1.728830,1.975930,0.900432,1.516197,0.980555,0.266476,0.043977,1.187522,True
4,4,Bangor and Holyhead,Wales,866.50,37673.91304,12.00,55.75,30.25,269.00,305.25,...,0.067290,0.720581,0.457670,1.256666,0.538422,1.097013,0.231594,0.022483,0.607110,False


In [7]:
creative_col = "All creative industries: Employment 2011-2014"
noncreative_col = "not_creative: Employment 2011-2014"

In [8]:
# Creative + non-creative employment = total employment
nesta["total_employment"] = (
    nesta[creative_col] + nesta[noncreative_col]
)

# TTWA creative employment share
nesta["creative_share"] = (
    nesta[creative_col] / nesta["total_employment"]
)

# England & Wales creative employment share
ew_creative_share = (
    nesta[creative_col].sum()
    / nesta["total_employment"].sum()
)

# Overall creative Location Quotient
nesta["creative_overall_LQ"] = (
    nesta["creative_share"] / ew_creative_share
)

nesta[
    [
        "ttwa",
        creative_col,
        "total_employment",
        "creative_share",
        "creative_overall_LQ"
    ]
].sort_values(
    "creative_overall_LQ",
    ascending=False
).head(20)

,ttwa,All creative industries: Employment 2011-2014,total_employment,creative_share,creative_overall_LQ
122,Peterborough,18654.00,1.946351e+05,0.095841,2.588011
128,Reading,31141.00,3.696301e+05,0.084249,2.274996
68,Guildford and Aldershot,38266.00,4.634438e+05,0.082569,2.229623
121,Penzance,1344.75,1.774414e+04,0.075786,2.046454
109,Newbury,5623.75,8.060708e+04,0.069767,1.883945
96,London,409073.00,6.424852e+06,0.063670,1.719306
139,Slough and Heathrow,65398.75,1.056289e+06,0.061914,1.671869
46,Colchester,6370.00,1.143361e+05,0.055713,1.504428
118,Oxford,17941.25,3.272731e+05,0.054820,1.480327
27,Brighton,6776.25,1.256578e+05,0.053926,1.456181


Two-sample t-tests

In [9]:
import pandas as pd
import numpy as np
from scipy import stats

In [10]:
analysis_df = census.merge(
    nesta[["ttwa", "creative_overall_LQ"]],
    left_on="TTWA11NM",
    right_on="ttwa",
    how="inner",
    validate="one_to_one"
)

print("Number of TTWAs:", len(analysis_df))

Number of TTWAs: 173


In [11]:
analysis_df["creative_group"] = np.where(
    analysis_df["creative_overall_LQ"] > 1,
    "High creative",
    "Low creative"
)

analysis_df["creative_group"].value_counts()

creative_group
Low creative     156
High creative     17
Name: count, dtype: int64

In [12]:
variables = {
    "Level 4+ qualifications (%)": "pct_level4_plus",
    "No qualifications (%)": "pct_no_quals",
    "Higher managerial/professional (%)": "pct_higher_managerial",
    "Routine occupations (%)": "pct_routine",
    "Self-employed (%)": "pct_self_employed",
    "Economically inactive (%)": "pct_econ_inactive",
    "Private rented (%)": "pct_private_rented",
    "Owned housing (%)": "pct_owned",
    "Population density": "pop_density",
    "Urban population (%)": "pct_urban",
    "Urban-rural score": "urban_rural_score",
    "Median house price (£)": "median_house_price"
}

In [13]:
results = []

for label, var in variables.items():

    
    temp = analysis_df[
        [var, "creative_overall_LQ", "creative_group"]
    ].dropna()

    # -----------------------------
    # Overall descriptive statistics
    # -----------------------------
    overall_mean = temp[var].mean()
    overall_sd = temp[var].std()
    overall_min = temp[var].min()
    overall_max = temp[var].max()

    # -----------------------------
    # Split into high / low creative
    # -----------------------------
    low = temp.loc[
        temp["creative_group"] == "Low creative",
        var
    ]

    high = temp.loc[
        temp["creative_group"] == "High creative",
        var
    ]

    low_mean = low.mean()
    high_mean = high.mean()

    # High creative minus low creative
    difference = high_mean - low_mean

    # -----------------------------
    # Welch two-sample t-test
    # -----------------------------
    t_stat, p_value = stats.ttest_ind(
        high,
        low,
        equal_var=False
    )

    # -----------------------------
    # Pearson correlation with
    # continuous creative LQ
    # -----------------------------
    r, corr_p = stats.pearsonr(
        temp[var],
        temp["creative_overall_LQ"]
    )

    # -----------------------------
    # Significance stars
    # -----------------------------
    if p_value < 0.001:
        stars = "***"
    elif p_value < 0.01:
        stars = "**"
    elif p_value < 0.05:
        stars = "*"
    else:
        stars = ""

    results.append({
        "Variable": label,
        "N": len(temp),
        "Mean": overall_mean,
        "SD": overall_sd,
        "Min": overall_min,
        "Max": overall_max,
        "Low creative mean": low_mean,
        "High creative mean": high_mean,
        "Difference": difference,
        "t": t_stat,
        "p": p_value,
        "Correlation": r,
        "Correlation p": corr_p,
        "Stars": stars
    })

results_df = pd.DataFrame(results)

results_df

,Variable,N,Mean,SD,Min,Max,Low creative mean,High creative mean,Difference,t,p,Correlation,Correlation p,Stars
0,Level 4+ qualifications (%),173,25.062312,5.004876,14.170000,37.190000,24.190705,33.060588,8.869883,8.651510,3.068448e-08,0.626078,3.234080e-20,***
1,No qualifications (%),173,23.969017,4.243589,15.690000,36.670000,24.605577,18.127647,-6.477930,-10.693427,1.159350e-11,-0.615582,2.015431e-19,***
2,Higher managerial/professional (%),173,9.174913,2.602442,4.580000,16.800000,8.684038,13.679412,4.995373,7.207887,1.054632e-06,0.697972,1.385027e-26,***
3,Routine occupations (%),173,12.012543,2.700492,6.800000,22.310000,12.378141,8.657647,-3.720494,-8.200770,1.553690e-08,-0.580032,6.129639e-17,***
4,Self-employed (%),173,15.812943,4.863457,8.378679,31.969483,15.777261,16.140382,0.363121,0.419787,6.781134e-01,0.083749,2.733052e-01,
5,Economically inactive (%),173,31.048337,3.472671,23.726267,40.992888,31.462977,27.243406,-4.219571,-6.207156,2.919088e-06,-0.437602,1.740071e-09,***
6,Private rented (%),173,15.558555,2.723500,8.730000,27.050000,15.398846,17.024118,1.625271,1.718214,1.033091e-01,0.211447,5.227950e-03,
7,Owned housing (%),173,67.398092,3.686802,49.420000,76.460000,67.562885,65.885882,-1.677002,-1.186407,2.515560e-01,-0.078551,3.042894e-01,
8,Population density,173,4.165446,4.828979,0.185492,36.870833,3.867661,6.898063,3.030402,1.315275,2.062658e-01,0.193447,1.076956e-02,
9,Urban population (%),173,63.069286,26.452238,0.000000,98.819520,62.117321,71.804960,9.687638,1.987188,5.809233e-02,0.135432,7.563169e-02,


In [14]:
results_df[
    [
        "Variable",
        "N",
        "Low creative mean",
        "High creative mean",
        "Difference",
        "t",
        "p",
        "Correlation",
        "Correlation p"
    ]
].round(4)

,Variable,N,Low creative mean,High creative mean,Difference,t,p,Correlation,Correlation p
0,Level 4+ qualifications (%),173,24.1907,33.0606,8.8699,8.6515,0.0000,0.6261,0.0000
1,No qualifications (%),173,24.6056,18.1276,-6.4779,-10.6934,0.0000,-0.6156,0.0000
2,Higher managerial/professional (%),173,8.6840,13.6794,4.9954,7.2079,0.0000,0.6980,0.0000
3,Routine occupations (%),173,12.3781,8.6576,-3.7205,-8.2008,0.0000,-0.5800,0.0000
4,Self-employed (%),173,15.7773,16.1404,0.3631,0.4198,0.6781,0.0837,0.2733
5,Economically inactive (%),173,31.4630,27.2434,-4.2196,-6.2072,0.0000,-0.4376,0.0000
6,Private rented (%),173,15.3988,17.0241,1.6253,1.7182,0.1033,0.2114,0.0052
7,Owned housing (%),173,67.5629,65.8859,-1.6770,-1.1864,0.2516,-0.0786,0.3043
8,Population density,173,3.8677,6.8981,3.0304,1.3153,0.2063,0.1934,0.0108
9,Urban population (%),173,62.1173,71.8050,9.6876,1.9872,0.0581,0.1354,0.0756


Final t-test table for erp project

In [15]:
appendix_table = results_df.copy()

def significance_stars(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


appendix_table["Stars"] = appendix_table["p"].apply(significance_stars)

appendix_table["Difference"] = (
    appendix_table["Difference"].map(lambda x: f"{x:.2f}")
    + appendix_table["Stars"]
)


for col in [
    "Mean",
    "SD",
    "Min",
    "Max",
    "Low creative mean",
    "High creative mean"
]:
    appendix_table[col] = appendix_table[col].round(2)


# Correlation to 3 decimal places
appendix_table["Correlation"] = (
    appendix_table["Correlation"].round(3)
)



appendix_table = appendix_table.rename(
    columns={
        "Low creative mean": "LQ ≤ 1 mean",
        "High creative mean": "LQ > 1 mean"
    }
)


appendix_table = appendix_table[
    [
        "Variable",
        "Mean",
        "SD",
        "Min",
        "Max",
        "LQ ≤ 1 mean",
        "LQ > 1 mean",
        "Difference",
        "Correlation"
    ]
]



appendix_table

,Variable,Mean,SD,Min,Max,LQ ≤ 1 mean,LQ > 1 mean,Difference,Correlation
0,Level 4+ qualifications (%),25.06,5.00,14.17,37.19,24.19,33.06,8.87***,0.626
1,No qualifications (%),23.97,4.24,15.69,36.67,24.61,18.13,-6.48***,-0.616
2,Higher managerial/professional (%),9.17,2.60,4.58,16.80,8.68,13.68,5.00***,0.698
3,Routine occupations (%),12.01,2.70,6.80,22.31,12.38,8.66,-3.72***,-0.580
4,Self-employed (%),15.81,4.86,8.38,31.97,15.78,16.14,0.36,0.084
5,Economically inactive (%),31.05,3.47,23.73,40.99,31.46,27.24,-4.22***,-0.438
6,Private rented (%),15.56,2.72,8.73,27.05,15.40,17.02,1.63,0.211
7,Owned housing (%),67.40,3.69,49.42,76.46,67.56,65.89,-1.68,-0.079
8,Population density,4.17,4.83,0.19,36.87,3.87,6.90,3.03,0.193
9,Urban population (%),63.07,26.45,0.00,98.82,62.12,71.80,9.69,0.135
